# 03 - Build `ml.trip_validity_trip_metrics`

One row per trip, reading **only** from `ml.trip_validity_trips`,
`ml.trip_validity_trip_fares`, `ml.trip_validity_route_gtfs_match`,
`ml.trip_validity_route_shapes`, and `ml.trip_validity_route_schedule` —
never `silver`. Built as a sequence of independently-committed `UPDATE`
stages (mirroring notebooks 01/02) rather than one giant `INSERT`, so a
bug in one stage doesn't force redoing the others.

Column-level provenance is documented via `COMMENT ON COLUMN` at the end,
same as the other two notebooks.

In [1]:
import os
from pathlib import Path

import psycopg
from psycopg import sql

In [2]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))

'/home/victor/repos/opa-database'

In [3]:
from opa_database.config import settings

conn = psycopg.connect(settings.db_dsn)
print("connected")

connected


In [4]:
conn.execute("""
    DROP TABLE IF EXISTS ml.trip_validity_trip_metrics CASCADE;

    CREATE TABLE ml.trip_validity_trip_metrics (
        trip_id  bigint PRIMARY KEY REFERENCES ml.trip_validity_trips (trip_id),
        route_id                             text NOT NULL,
        route_direction                      integer NOT NULL,
        trip_duration_seconds                bigint,
        trip_fare_count                      integer NOT NULL,

        trip_distance_meters                 double precision,
        trip_points_standard_distance_meters double precision,

        fare_gap_avg_seconds                 double precision,
        fare_gap_stddev_seconds              double precision,
        fare_span_seconds                    double precision,

        gtfs_feed_version_date               date,
        gtfs_route_short_name                text,
        gtfs_route_has_both_directions        boolean,
        gtfs_shape_id_i                      text,
        gtfs_shape_id_v                      text,

        trip_start_distance_to_i_start_meters double precision,
        trip_start_distance_to_v_start_meters double precision,
        trip_end_distance_to_i_end_meters     double precision,
        trip_end_distance_to_v_end_meters     double precision,
        path_frechet_distance_to_i_meters     double precision,
        path_frechet_distance_to_v_meters     double precision,
        path_hausdorff_distance_to_i_meters   double precision,
        path_hausdorff_distance_to_v_meters   double precision,

        route_i_length_meters                double precision,
        route_v_length_meters                double precision,

        route_i_scheduled_duration_avg_seconds         double precision,
        route_i_scheduled_duration_n_trips             integer,
        route_v_scheduled_duration_avg_seconds         double precision,
        route_v_scheduled_duration_n_trips             integer,
        route_i_scheduled_duration_avg_seconds_at_hour double precision,
        route_i_scheduled_duration_n_trips_at_hour     integer,
        route_v_scheduled_duration_avg_seconds_at_hour double precision,
        route_v_scheduled_duration_n_trips_at_hour     integer,

        route_avg_trip_duration_seconds_loo                 double precision,
        route_avg_trip_duration_n_trips_loo                 integer,
        route_direction_avg_trip_duration_seconds_loo       double precision,
        route_direction_avg_trip_duration_n_trips_loo       integer,
        route_hour_avg_trip_duration_seconds_loo            double precision,
        route_hour_avg_trip_duration_n_trips_loo            integer,
        route_direction_hour_avg_trip_duration_seconds_loo  double precision,
        route_direction_hour_avg_trip_duration_n_trips_loo  integer
    );
""")

conn.execute("""
    INSERT INTO ml.trip_validity_trip_metrics (
        trip_id, route_id, route_direction, trip_duration_seconds, trip_fare_count
    )
    SELECT trip_id, route_id, route_direction, trip_duration_seconds,
           trip_fare_count
    FROM ml.trip_validity_trips;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.trip_validity_trip_metrics;")
    print("base rows:", cur.fetchone()[0])

base rows: 940988


## Stage 1 - trip geometry: distance traveled and point cohesion

`trip_points_standard_distance_meters` is the spatial-statistics analogue
of standard deviation: RMS distance of every geo-tagged fare from the
trip's own centroid. A trip with a single geo-tagged fare gets `0` (its
one point has zero distance from itself as centroid) — mathematically
valid, not an error case. `NULL` only when a trip has zero geo-tagged
fares (nothing to compute).

In [5]:
conn.execute("""
    UPDATE ml.trip_validity_trip_metrics tm
    SET trip_distance_meters = ST_Length(t.trip_path::geography)
    FROM ml.trip_validity_trips t
    WHERE t.trip_id = tm.trip_id;
""")

conn.execute("""
    WITH trip_centroids AS (
        SELECT trip_id, ST_Centroid(ST_Collect(geom)) AS centroid
        FROM ml.trip_validity_trip_fares
        WHERE geom IS NOT NULL
        GROUP BY trip_id
    ),
    trip_cohesion AS (
        SELECT f.trip_id,
               sqrt(avg(
                   ST_Distance(f.geom::geography, c.centroid::geography) ^ 2
               )) AS standard_distance
        FROM ml.trip_validity_trip_fares f
        JOIN trip_centroids c ON c.trip_id = f.trip_id
        WHERE f.geom IS NOT NULL
        GROUP BY f.trip_id
    )
    UPDATE ml.trip_validity_trip_metrics tm
    SET trip_points_standard_distance_meters = tc.standard_distance
    FROM trip_cohesion tc
    WHERE tc.trip_id = tm.trip_id;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT count(*) FILTER (WHERE trip_distance_meters IS NOT NULL),
               count(*) FILTER (WHERE trip_points_standard_distance_meters IS NOT NULL)
        FROM ml.trip_validity_trip_metrics;
    """)
    print("non-null trip_distance / cohesion:", cur.fetchone())

non-null trip_distance / cohesion: (899299, 899299)


## Stage 2 - fare timing (all fares, not just geo-tagged)

`fare_gap_avg_seconds`/`fare_gap_stddev_seconds` are `NULL` for a
1-fare trip (no gap exists to average). `fare_span_seconds` is `0`
(not `NULL`) for a 1-fare trip — a well-defined zero-length span.

In [6]:
conn.execute("""
    WITH ordered_fares AS (
        SELECT trip_id, boarding_at,
               boarding_at - LAG(boarding_at)
                   OVER (PARTITION BY trip_id ORDER BY boarding_at) AS gap
        FROM ml.trip_validity_trip_fares
    ),
    fare_stats AS (
        SELECT trip_id,
               avg(EXTRACT(EPOCH FROM gap)) AS fare_gap_avg_seconds,
               stddev(EXTRACT(EPOCH FROM gap)) AS fare_gap_stddev_seconds,
               EXTRACT(EPOCH FROM (max(boarding_at) - min(boarding_at)))
                   AS fare_span_seconds
        FROM ordered_fares
        GROUP BY trip_id
    )
    UPDATE ml.trip_validity_trip_metrics tm
    SET fare_gap_avg_seconds = fs.fare_gap_avg_seconds,
        fare_gap_stddev_seconds = fs.fare_gap_stddev_seconds,
        fare_span_seconds = fs.fare_span_seconds
    FROM fare_stats fs
    WHERE fs.trip_id = tm.trip_id;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT count(*) FILTER (WHERE fare_gap_avg_seconds IS NOT NULL),
               count(*) FILTER (WHERE fare_span_seconds = 0),
               count(*) FILTER (WHERE trip_fare_count = 1)
        FROM ml.trip_validity_trip_metrics;
    """)
    print("non-null gap avg / zero span / 1-fare trips:", cur.fetchone())

non-null gap avg / zero span / 1-fare trips: (653416, 287572, 287572)


## Stage 3 - GTFS resolution passthrough

Straight copy from `ml.trip_validity_route_gtfs_match`, joined via
`(route_id, trip_date)` (through `ml.trip_validity_trips` for
`trip_date`, which isn't stored on this table). Everything downstream
that needs the resolved feed/shape reads it from here rather than
rejoining `route_gtfs_match` again.

In [7]:
conn.execute("""
    UPDATE ml.trip_validity_trip_metrics tm
    SET gtfs_feed_version_date = m.gtfs_feed_version_date,
        gtfs_route_short_name = m.gtfs_route_short_name,
        gtfs_route_has_both_directions = m.gtfs_route_has_both_directions,
        gtfs_shape_id_i = m.gtfs_shape_id_i,
        gtfs_shape_id_v = m.gtfs_shape_id_v
    FROM ml.trip_validity_trips t
    JOIN ml.trip_validity_route_gtfs_match m
      ON m.route_id = t.route_id AND m.trip_date = t.trip_date
    WHERE t.trip_id = tm.trip_id;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT count(*) FILTER (WHERE gtfs_feed_version_date IS NOT NULL),
               count(*) FILTER (WHERE gtfs_route_has_both_directions)
        FROM ml.trip_validity_trip_metrics;
    """)
    print("trips with resolved feed / with both directions:", cur.fetchone())

trips with resolved feed / with both directions: (934155, 891158)


## Stage 4 - path-vs-route comparison (I and V, regardless of the row's own direction)

Joins in `ml.trip_validity_route_shapes` via the shape ids set in Stage
3. `trip_path` is projected to `EPSG:31984` **once per row**, in its own
CTE, and reused for all four Frechet/Hausdorff calls below it — an
earlier version of this cell repeated the `ST_Transform` call inline for
each of the four, which Postgres does not automatically deduplicate
across a target list, quadrupling that part of the work for no reason.
`ST_FrechetDistance`/`ST_HausdorffDistance` have no `geography`
overload, which is why the projection is needed at all.

All eight columns are `NULL` whenever `trip_path` is `NULL` or the
corresponding shape doesn't exist for this route — PostGIS's distance
functions are strict (`NULL` in, `NULL` out), so no extra `CASE` guards
are needed.

This is the most compute-heavy stage by far: up to ~1.8M Frechet +
Hausdorff evaluations against shapes with ~100-300 points each. A
2,000-row timing sample came out to ~430μs/row, i.e. roughly 6-7 minutes
for all 899,299 trips with a path — there's no index that can accelerate
an arbitrary pairwise curve-distance calculation (unlike the GiST
indexes used elsewhere in this project for spatial *filtering*), so this
is just the real cost of the computation, not a stuck or misconfigured
query.

In [8]:
conn.execute("""
    WITH trip_paths AS (
        SELECT tm2.trip_id, t.trip_path,
               ST_Transform(t.trip_path, 31984) AS trip_path_metric
        FROM ml.trip_validity_trip_metrics tm2
        JOIN ml.trip_validity_trips t ON t.trip_id = tm2.trip_id
        WHERE t.trip_path IS NOT NULL
    ),
    path_compare AS (
        SELECT
            tp.trip_id,
            ST_Distance(
                ST_StartPoint(tp.trip_path)::geography,
                ST_StartPoint(si.shape_geom)::geography
            ) AS d_start_i,
            ST_Distance(
                ST_StartPoint(tp.trip_path)::geography,
                ST_StartPoint(sv.shape_geom)::geography
            ) AS d_start_v,
            ST_Distance(
                ST_EndPoint(tp.trip_path)::geography,
                ST_EndPoint(si.shape_geom)::geography
            ) AS d_end_i,
            ST_Distance(
                ST_EndPoint(tp.trip_path)::geography,
                ST_EndPoint(sv.shape_geom)::geography
            ) AS d_end_v,
            ST_FrechetDistance(tp.trip_path_metric, si.shape_geom_metric) AS frechet_i,
            ST_FrechetDistance(tp.trip_path_metric, sv.shape_geom_metric) AS frechet_v,
            ST_HausdorffDistance(tp.trip_path_metric, si.shape_geom_metric)
                AS hausdorff_i,
            ST_HausdorffDistance(tp.trip_path_metric, sv.shape_geom_metric)
                AS hausdorff_v
        FROM trip_paths tp
        JOIN ml.trip_validity_trip_metrics tm2 ON tm2.trip_id = tp.trip_id
        LEFT JOIN ml.trip_validity_route_shapes si
          ON si.feed_version_date = tm2.gtfs_feed_version_date
         AND si.shape_id = tm2.gtfs_shape_id_i
        LEFT JOIN ml.trip_validity_route_shapes sv
          ON sv.feed_version_date = tm2.gtfs_feed_version_date
         AND sv.shape_id = tm2.gtfs_shape_id_v
    )
    UPDATE ml.trip_validity_trip_metrics tm
    SET trip_start_distance_to_i_start_meters = x.d_start_i,
        trip_start_distance_to_v_start_meters = x.d_start_v,
        trip_end_distance_to_i_end_meters = x.d_end_i,
        trip_end_distance_to_v_end_meters = x.d_end_v,
        path_frechet_distance_to_i_meters = x.frechet_i,
        path_frechet_distance_to_v_meters = x.frechet_v,
        path_hausdorff_distance_to_i_meters = x.hausdorff_i,
        path_hausdorff_distance_to_v_meters = x.hausdorff_v
    FROM path_compare x
    WHERE x.trip_id = tm.trip_id;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT count(*) FILTER (WHERE path_frechet_distance_to_i_meters IS NOT NULL),
               count(*) FILTER (WHERE path_frechet_distance_to_v_meters IS NOT NULL)
        FROM ml.trip_validity_trip_metrics;
    """)
    print("non-null frechet-to-I / frechet-to-V:", cur.fetchone())

non-null frechet-to-I / frechet-to-V: (873954, 877493)


## Stage 5 - route length and scheduled-duration aggregates

`route_i/v_length_meters` come straight from `route_shapes`.
Scheduled-duration aggregates are computed once over
`ml.trip_validity_route_schedule` (grouped by `(feed, gtfs_route_id,
direction)` and, separately, `(feed, gtfs_route_id, direction,
start_hour)`), then joined onto every row — not recomputed per row.
Each average carries its own row-count column
(`..._n_trips[_at_hour]`); `NULL` when no scheduled GTFS trips exist for
that combination.

In [9]:
conn.execute("""
    UPDATE ml.trip_validity_trip_metrics tm
    SET route_i_length_meters = si.shape_length_meters,
        route_v_length_meters = sv.shape_length_meters
    FROM ml.trip_validity_trip_metrics tm2
    LEFT JOIN ml.trip_validity_route_shapes si
      ON si.feed_version_date = tm2.gtfs_feed_version_date
     AND si.shape_id = tm2.gtfs_shape_id_i
    LEFT JOIN ml.trip_validity_route_shapes sv
      ON sv.feed_version_date = tm2.gtfs_feed_version_date
     AND sv.shape_id = tm2.gtfs_shape_id_v
    WHERE tm2.trip_id = tm.trip_id;
""")

conn.execute("""
    WITH schedule_overall AS (
        SELECT feed_version_date, gtfs_route_id, direction,
               avg(scheduled_duration_seconds) AS avg_dur, count(*) AS n
        FROM ml.trip_validity_route_schedule
        GROUP BY feed_version_date, gtfs_route_id, direction
    ),
    schedule_hour AS (
        SELECT feed_version_date, gtfs_route_id, direction, start_hour,
               avg(scheduled_duration_seconds) AS avg_dur, count(*) AS n
        FROM ml.trip_validity_route_schedule
        GROUP BY feed_version_date, gtfs_route_id, direction, start_hour
    ),
    row_context AS (
        SELECT tm2.trip_id, tm2.gtfs_feed_version_date, t.trip_hour, m.gtfs_route_id
        FROM ml.trip_validity_trip_metrics tm2
        JOIN ml.trip_validity_trips t ON t.trip_id = tm2.trip_id
        JOIN ml.trip_validity_route_gtfs_match m
          ON m.route_id = t.route_id AND m.trip_date = t.trip_date
    ),
    sched_join AS (
        SELECT
            rc.trip_id,
            so_i.avg_dur AS route_i_avg, so_i.n AS route_i_n,
            so_v.avg_dur AS route_v_avg, so_v.n AS route_v_n,
            sh_i.avg_dur AS route_i_avg_hr, sh_i.n AS route_i_n_hr,
            sh_v.avg_dur AS route_v_avg_hr, sh_v.n AS route_v_n_hr
        FROM row_context rc
        LEFT JOIN schedule_overall so_i
          ON so_i.feed_version_date = rc.gtfs_feed_version_date
         AND so_i.gtfs_route_id = rc.gtfs_route_id AND so_i.direction = 'I'
        LEFT JOIN schedule_overall so_v
          ON so_v.feed_version_date = rc.gtfs_feed_version_date
         AND so_v.gtfs_route_id = rc.gtfs_route_id AND so_v.direction = 'V'
        LEFT JOIN schedule_hour sh_i
          ON sh_i.feed_version_date = rc.gtfs_feed_version_date
         AND sh_i.gtfs_route_id = rc.gtfs_route_id AND sh_i.direction = 'I'
         AND sh_i.start_hour = rc.trip_hour
        LEFT JOIN schedule_hour sh_v
          ON sh_v.feed_version_date = rc.gtfs_feed_version_date
         AND sh_v.gtfs_route_id = rc.gtfs_route_id AND sh_v.direction = 'V'
         AND sh_v.start_hour = rc.trip_hour
    )
    UPDATE ml.trip_validity_trip_metrics tm
    SET route_i_scheduled_duration_avg_seconds = sj.route_i_avg,
        route_i_scheduled_duration_n_trips = sj.route_i_n,
        route_v_scheduled_duration_avg_seconds = sj.route_v_avg,
        route_v_scheduled_duration_n_trips = sj.route_v_n,
        route_i_scheduled_duration_avg_seconds_at_hour = sj.route_i_avg_hr,
        route_i_scheduled_duration_n_trips_at_hour = sj.route_i_n_hr,
        route_v_scheduled_duration_avg_seconds_at_hour = sj.route_v_avg_hr,
        route_v_scheduled_duration_n_trips_at_hour = sj.route_v_n_hr
    FROM sched_join sj
    WHERE sj.trip_id = tm.trip_id;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT count(*) FILTER (WHERE route_i_length_meters IS NOT NULL),
               count(*) FILTER (
                   WHERE route_i_scheduled_duration_avg_seconds IS NOT NULL
               ),
               count(*) FILTER (
                   WHERE route_i_scheduled_duration_avg_seconds_at_hour IS NOT NULL
               )
        FROM ml.trip_validity_trip_metrics;
    """)
    print("non-null route_i_length / sched_avg / sched_avg_at_hour:", cur.fetchone())

non-null route_i_length / sched_avg / sched_avg_at_hour: (910838, 910838, 893070)


## Stage 6 - leave-one-out historical trip-duration aggregates

For each of the four groupings (route / route+direction / route+hour /
route+direction+hour), the average `trip_duration_seconds` **excludes
the row's own value** — computed via `SUM`/`COUNT` window functions over
the whole group, then subtracting the row's own contribution before
dividing. `COUNT`/`SUM` already skip `NULL` durations (the 9
Delphi-sentinel trips), so a row with a `NULL` own-duration has nothing
to subtract — its LOO average is just the plain group average.

`NULL` (not `0` or an error) whenever removing the row's own
contribution leaves zero other trips in that group — e.g. a route with
only one trip all month. Each average has a matching `..._n_trips_loo`
column recording exactly how many *other* trips it was computed from, so
`NULL` vs. "computed from 1 trip" vs. "computed from 500 trips" stays
distinguishable downstream.

This is unrelated to train/test leakage (no split exists yet on this
dataset) — it's purely about not letting a trip's own duration count
toward "the average duration for trips like this," at the user's
request.

In [10]:
conn.execute("""
    WITH loo AS (
        SELECT
            trip_id, trip_duration_seconds,
            SUM(trip_duration_seconds) OVER (PARTITION BY route_id) AS route_sum,
            COUNT(trip_duration_seconds) OVER (PARTITION BY route_id) AS route_n,
            SUM(trip_duration_seconds)
                OVER (PARTITION BY route_id, route_direction) AS route_dir_sum,
            COUNT(trip_duration_seconds)
                OVER (PARTITION BY route_id, route_direction) AS route_dir_n,
            SUM(trip_duration_seconds)
                OVER (PARTITION BY route_id, trip_hour) AS route_hour_sum,
            COUNT(trip_duration_seconds)
                OVER (PARTITION BY route_id, trip_hour) AS route_hour_n,
            SUM(trip_duration_seconds)
                OVER (PARTITION BY route_id, route_direction, trip_hour)
                AS route_dir_hour_sum,
            COUNT(trip_duration_seconds)
                OVER (PARTITION BY route_id, route_direction, trip_hour)
                AS route_dir_hour_n
        FROM ml.trip_validity_trips
    ),
    loo_calc AS (
        SELECT
            trip_id,
            CASE WHEN trip_duration_seconds IS NULL
                 THEN CASE WHEN route_n > 0
                           THEN route_sum::double precision / route_n END
                 ELSE CASE WHEN route_n > 1
                           THEN (route_sum - trip_duration_seconds)
                                ::double precision / (route_n - 1)
                      END
            END AS route_avg_loo,
            CASE WHEN trip_duration_seconds IS NULL
                 THEN route_n ELSE route_n - 1 END AS route_n_loo,

            CASE WHEN trip_duration_seconds IS NULL
                 THEN CASE WHEN route_dir_n > 0
                           THEN route_dir_sum::double precision / route_dir_n END
                 ELSE CASE WHEN route_dir_n > 1
                           THEN (route_dir_sum - trip_duration_seconds)
                                ::double precision / (route_dir_n - 1)
                      END
            END AS route_dir_avg_loo,
            CASE WHEN trip_duration_seconds IS NULL
                 THEN route_dir_n ELSE route_dir_n - 1 END AS route_dir_n_loo,

            CASE WHEN trip_duration_seconds IS NULL
                 THEN CASE WHEN route_hour_n > 0
                           THEN route_hour_sum::double precision / route_hour_n END
                 ELSE CASE WHEN route_hour_n > 1
                           THEN (route_hour_sum - trip_duration_seconds)
                                ::double precision / (route_hour_n - 1)
                      END
            END AS route_hour_avg_loo,
            CASE WHEN trip_duration_seconds IS NULL
                 THEN route_hour_n ELSE route_hour_n - 1 END AS route_hour_n_loo,

            CASE WHEN trip_duration_seconds IS NULL
                 THEN CASE WHEN route_dir_hour_n > 0
                           THEN route_dir_hour_sum::double precision
                                / route_dir_hour_n END
                 ELSE CASE WHEN route_dir_hour_n > 1
                           THEN (route_dir_hour_sum - trip_duration_seconds)
                                ::double precision / (route_dir_hour_n - 1)
                      END
            END AS route_dir_hour_avg_loo,
            CASE WHEN trip_duration_seconds IS NULL
                 THEN route_dir_hour_n ELSE route_dir_hour_n - 1 END
                 AS route_dir_hour_n_loo
        FROM loo
    )
    UPDATE ml.trip_validity_trip_metrics tm
    SET route_avg_trip_duration_seconds_loo = lc.route_avg_loo,
        route_avg_trip_duration_n_trips_loo = lc.route_n_loo,
        route_direction_avg_trip_duration_seconds_loo = lc.route_dir_avg_loo,
        route_direction_avg_trip_duration_n_trips_loo = lc.route_dir_n_loo,
        route_hour_avg_trip_duration_seconds_loo = lc.route_hour_avg_loo,
        route_hour_avg_trip_duration_n_trips_loo = lc.route_hour_n_loo,
        route_direction_hour_avg_trip_duration_seconds_loo = lc.route_dir_hour_avg_loo,
        route_direction_hour_avg_trip_duration_n_trips_loo = lc.route_dir_hour_n_loo
    FROM loo_calc lc
    WHERE lc.trip_id = tm.trip_id;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            count(*) FILTER (
                WHERE route_avg_trip_duration_seconds_loo IS NULL
            ) AS route_loo_null,
            count(*) FILTER (
                WHERE route_direction_hour_avg_trip_duration_seconds_loo IS NULL
            ) AS finest_loo_null,
            min(route_avg_trip_duration_n_trips_loo) AS min_n
        FROM ml.trip_validity_trip_metrics;
    """)
    print(
        "route-level LOO nulls / finest-grain LOO nulls / min n used:",
        cur.fetchone(),
    )

route-level LOO nulls / finest-grain LOO nulls / min n used: (8, 238, 0)


## Stage 7 - directional progress correlation

For each trip, does its sequence of geo-tagged fares move steadily
*forward* along the official route shape as time passes, *backward*, or
neither? Computed as the Pearson correlation between each point's
position along the shape (`ST_LineLocatePoint`, a 0-to-1 fraction from
the shape's start to its end) and the point's timestamp
(`boarding_at`), across the trip's geo-tagged fares - one correlation
against the `-I` shape, one against `-V`, from the same set of points.

`+1` = perfect steady forward progress along that shape. `-1` = perfect
steady progress in reverse. Near `0` = no consistent relationship
(stationary, erratic, wrong route, or GPS noise dominating).

`trip_progress_correlation_n_points` records how many geo-tagged points
went into *both* correlations (same input points for I and V, only the
projection target differs) - a correlation from 2 points is always a
meaningless exact ±1 (a line through 2 points is always "perfectly
correlated"), so this column exists specifically to let you judge how
much to trust the correlation value next to it. Both correlations are
also `NULL` on their own if the corresponding shape (`gtfs_shape_id_i`/
`_v`) was never matched - already visible from that column, not
duplicated here.

In [11]:
conn.execute("""
    ALTER TABLE ml.trip_validity_trip_metrics
        ADD COLUMN trip_progress_correlation_to_i double precision,
        ADD COLUMN trip_progress_correlation_to_v double precision,
        ADD COLUMN trip_progress_correlation_n_points integer;
""")

conn.execute("""
    WITH point_stats AS (
        SELECT
            f.trip_id,
            count(*) AS n_points,
            corr(
                ST_LineLocatePoint(si.shape_geom, f.geom),
                extract(epoch FROM f.boarding_at)
            ) AS corr_i,
            corr(
                ST_LineLocatePoint(sv.shape_geom, f.geom),
                extract(epoch FROM f.boarding_at)
            ) AS corr_v
        FROM ml.trip_validity_trip_fares f
        JOIN ml.trip_validity_trip_metrics tm2 ON tm2.trip_id = f.trip_id
        LEFT JOIN ml.trip_validity_route_shapes si
          ON si.feed_version_date = tm2.gtfs_feed_version_date
         AND si.shape_id = tm2.gtfs_shape_id_i
        LEFT JOIN ml.trip_validity_route_shapes sv
          ON sv.feed_version_date = tm2.gtfs_feed_version_date
         AND sv.shape_id = tm2.gtfs_shape_id_v
        WHERE f.geom IS NOT NULL
        GROUP BY f.trip_id
    )
    UPDATE ml.trip_validity_trip_metrics tm
    SET trip_progress_correlation_to_i = ps.corr_i,
        trip_progress_correlation_to_v = ps.corr_v,
        trip_progress_correlation_n_points = ps.n_points
    FROM point_stats ps
    WHERE ps.trip_id = tm.trip_id;
""")
conn.commit()

MIN_RELIABLE_POINTS = 2  # a correlation from <=2 points is a meaningless exact +-1

with conn.cursor() as cur:
    cur.execute(
        """
        SELECT count(*) FILTER (WHERE trip_progress_correlation_to_i IS NOT NULL),
               count(*) FILTER (WHERE trip_progress_correlation_to_v IS NOT NULL),
               count(*) FILTER (WHERE trip_progress_correlation_n_points <= %(n)s)
        FROM ml.trip_validity_trip_metrics;
        """,
        {"n": MIN_RELIABLE_POINTS},
    )
    print(
        "non-null corr_i / corr_v / trips with <=2 points (unreliable):",
        cur.fetchone(),
    )

non-null corr_i / corr_v / trips with <=2 points (unreliable): (577407, 586638, 323385)


## Stage 8 - reverse-direction historical trip-duration averages

Companions to Stage 6's leave-one-out columns, but comparing this trip
against trips on the same route going the *opposite* `route_direction`
instead of the same one. Not `_loo`-suffixed since there's nothing to
leave out - a trip in direction `0` can never appear in direction `1`'s
group in the first place, so this is a plain average, not a
self-exclusion. `NULL` (with `n_trips = NULL`, not `0`) when the route
has no trips at all going the reverse direction (e.g. a one-way-only
route).

In [12]:
conn.execute("""
    ALTER TABLE ml.trip_validity_trip_metrics
        ADD COLUMN route_reverse_direction_avg_trip_duration_seconds
            double precision,
        ADD COLUMN route_reverse_direction_n_trips integer,
        ADD COLUMN route_reverse_direction_hour_avg_trip_duration_seconds
            double precision,
        ADD COLUMN route_reverse_direction_hour_n_trips integer;
""")

conn.execute("""
    WITH dir_stats AS (
        SELECT route_id, route_direction,
               avg(trip_duration_seconds) AS avg_dur,
               count(trip_duration_seconds) AS n
        FROM ml.trip_validity_trips
        GROUP BY route_id, route_direction
    ),
    dir_hour_stats AS (
        SELECT route_id, route_direction, trip_hour,
               avg(trip_duration_seconds) AS avg_dur,
               count(trip_duration_seconds) AS n
        FROM ml.trip_validity_trips
        GROUP BY route_id, route_direction, trip_hour
    )
    UPDATE ml.trip_validity_trip_metrics tm
    SET route_reverse_direction_avg_trip_duration_seconds = ds.avg_dur,
        route_reverse_direction_n_trips = ds.n,
        route_reverse_direction_hour_avg_trip_duration_seconds = dhs.avg_dur,
        route_reverse_direction_hour_n_trips = dhs.n
    FROM ml.trip_validity_trips t
    LEFT JOIN dir_stats ds
      ON ds.route_id = t.route_id
     AND ds.route_direction = (
         CASE WHEN t.route_direction = 0 THEN 1 WHEN t.route_direction = 1 THEN 0 END
     )
    LEFT JOIN dir_hour_stats dhs
      ON dhs.route_id = t.route_id
     AND dhs.route_direction = (
         CASE WHEN t.route_direction = 0 THEN 1 WHEN t.route_direction = 1 THEN 0 END
     )
     AND dhs.trip_hour = t.trip_hour
    WHERE t.trip_id = tm.trip_id;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT count(*) FILTER (
                   WHERE route_reverse_direction_avg_trip_duration_seconds IS NOT NULL
               ),
               count(*) FILTER (
                   WHERE route_reverse_direction_hour_avg_trip_duration_seconds
                       IS NOT NULL
               )
        FROM ml.trip_validity_trip_metrics;
    """)
    print(
        "non-null reverse-direction avg / reverse-direction+hour avg:",
        cur.fetchone(),
    )

non-null reverse-direction avg / reverse-direction+hour avg: (897435, 891548)


## Column-level provenance comments

In [13]:
def comment_on_column(cur: psycopg.Cursor, table: str, col: str, text: str) -> None:
    """Apply a COMMENT ON COLUMN for one column via safe SQL composition."""
    cur.execute(
        sql.SQL("COMMENT ON COLUMN ml.{}.{} IS {};").format(
            sql.Identifier(table), sql.Identifier(col), sql.Literal(text)
        )
    )


METRICS_COMMENTS = {
    "trip_id": "= ml.trip_validity_trips.trip_id (FK, PK here too).",
    "route_id": "Copied from ml.trip_validity_trips.route_id.",
    "route_direction": (
        "Copied from ml.trip_validity_trips.route_direction (this trip's "
        "own observed AFC direction)."
    ),
    "trip_duration_seconds": (
        "Copied from ml.trip_validity_trips.trip_duration_seconds."
    ),
    "trip_fare_count": "Copied from ml.trip_validity_trips.trip_fare_count.",
    "trip_distance_meters": (
        "ST_Length(trip_path::geography) from ml.trip_validity_trips. "
        "NULL if trip_path is NULL."
    ),
    "trip_points_standard_distance_meters": (
        "RMS distance of this trip's geo-tagged fares "
        "(ml.trip_validity_trip_fares) from their own centroid. 0 for a "
        "single geo-tagged fare, NULL for zero."
    ),
    "fare_gap_avg_seconds": (
        "Mean gap between consecutive boarding_at values (ALL fares, "
        "ml.trip_validity_trip_fares), ordered by boarding_at. NULL if "
        "only 1 fare."
    ),
    "fare_gap_stddev_seconds": (
        "Sample stddev of the same gaps. NULL if fewer than 2 gaps (i.e. "
        "fewer than 3 fares) exist."
    ),
    "fare_span_seconds": (
        "max(boarding_at) - min(boarding_at) across all fares. 0 (not "
        "NULL) for a 1-fare trip."
    ),
    "gtfs_feed_version_date": (
        "From ml.trip_validity_route_gtfs_match, resolved by (route_id, trip_date)."
    ),
    "gtfs_route_short_name": (
        "From ml.trip_validity_route_gtfs_match - the actual GTFS join "
        "key used, stored for traceability."
    ),
    "gtfs_route_has_both_directions": (
        "From ml.trip_validity_route_gtfs_match. NULL means the route "
        "never matched any GTFS feed at all; false means it matched but "
        "only one direction has a shape; true means both directions "
        "have a shape."
    ),
    "gtfs_shape_id_i": "From ml.trip_validity_route_gtfs_match.",
    "gtfs_shape_id_v": "From ml.trip_validity_route_gtfs_match.",
    "trip_start_distance_to_i_start_meters": (
        "ST_Distance (geography) between ST_StartPoint(trip_path) and "
        "ST_StartPoint of the matched -I shape's geometry "
        "(ml.trip_validity_route_shapes). Regardless of this trip's own "
        "route_direction."
    ),
    "trip_start_distance_to_v_start_meters": "Same, against the -V shape.",
    "trip_end_distance_to_i_end_meters": (
        "ST_Distance (geography) between ST_EndPoint(trip_path) and "
        "ST_EndPoint of the matched -I shape's geometry."
    ),
    "trip_end_distance_to_v_end_meters": "Same, against the -V shape.",
    "path_frechet_distance_to_i_meters": (
        "ST_FrechetDistance between trip_path and the -I shape, both "
        "projected to EPSG:31984 (SIRGAS2000/UTM 24S) first (no "
        "geography overload exists for this function)."
    ),
    "path_frechet_distance_to_v_meters": "Same, against the -V shape.",
    "path_hausdorff_distance_to_i_meters": (
        "ST_HausdorffDistance between trip_path and the -I shape, both "
        "projected to EPSG:31984 first."
    ),
    "path_hausdorff_distance_to_v_meters": "Same, against the -V shape.",
    "route_i_length_meters": (
        "ml.trip_validity_route_shapes.shape_length_meters for the matched -I shape."
    ),
    "route_v_length_meters": "Same, for the matched -V shape.",
    "route_i_scheduled_duration_avg_seconds": (
        "avg(scheduled_duration_seconds) over "
        "ml.trip_validity_route_schedule for this row's resolved (feed, "
        "gtfs_route_id, direction='I'). NULL if no scheduled GTFS trips "
        "exist for it."
    ),
    "route_i_scheduled_duration_n_trips": "Row count backing the average above.",
    "route_v_scheduled_duration_avg_seconds": (
        "Same as route_i_scheduled_duration_avg_seconds, direction='V'."
    ),
    "route_v_scheduled_duration_n_trips": "Row count backing the average above.",
    "route_i_scheduled_duration_avg_seconds_at_hour": (
        "Same as route_i_scheduled_duration_avg_seconds, additionally "
        "filtered to GTFS scheduled trips whose start_hour matches this "
        "row's trip_hour."
    ),
    "route_i_scheduled_duration_n_trips_at_hour": (
        "Row count backing the average above."
    ),
    "route_v_scheduled_duration_avg_seconds_at_hour": (
        "Same as route_i_scheduled_duration_avg_seconds_at_hour, direction='V'."
    ),
    "route_v_scheduled_duration_n_trips_at_hour": (
        "Row count backing the average above."
    ),
    "route_avg_trip_duration_seconds_loo": (
        "Leave-one-out mean trip_duration_seconds across all "
        "ml.trip_validity_trips sharing this row's route_id (this row's "
        "own value excluded). NULL if no other trips exist in the group."
    ),
    "route_avg_trip_duration_n_trips_loo": (
        "Count of other trips the average above was computed from (0 if "
        "none - explains a NULL average)."
    ),
    "route_direction_avg_trip_duration_seconds_loo": (
        "Same, grouped by (route_id, route_direction)."
    ),
    "route_direction_avg_trip_duration_n_trips_loo": (
        "Count backing the average above."
    ),
    "route_hour_avg_trip_duration_seconds_loo": (
        "Same, grouped by (route_id, trip_hour)."
    ),
    "route_hour_avg_trip_duration_n_trips_loo": "Count backing the average above.",
    "route_direction_hour_avg_trip_duration_seconds_loo": (
        "Same, grouped by (route_id, route_direction, trip_hour)."
    ),
    "route_direction_hour_avg_trip_duration_n_trips_loo": (
        "Count backing the average above."
    ),
    "trip_progress_correlation_to_i": (
        "Pearson correlation (corr()) between ST_LineLocatePoint(-I "
        "shape, point) and extract(epoch from boarding_at), across this "
        "trip's geo-tagged fares in ml.trip_validity_trip_fares. +1 = "
        "steady forward progress along the -I shape as time passes, -1 "
        "= steady progress in reverse, ~0 = no consistent relationship. "
        "NULL if gtfs_shape_id_i is NULL or fewer than 2 geo-tagged "
        "fares exist."
    ),
    "trip_progress_correlation_to_v": (
        "Same as trip_progress_correlation_to_i, against the -V shape."
    ),
    "trip_progress_correlation_n_points": (
        "Count of geo-tagged fares used for BOTH correlations above "
        "(same input points, different projection target). A value of "
        "2 means the correlation is a meaningless exact +-1 (a line "
        "through 2 points is always perfectly correlated) - use this to "
        "judge how much to trust the two correlation columns."
    ),
    "route_reverse_direction_avg_trip_duration_seconds": (
        "avg(trip_duration_seconds) across ml.trip_validity_trips "
        "sharing this row's route_id but with the OPPOSITE "
        "route_direction (0<->1). Not leave-one-out - this trip can "
        "never be a member of that group. NULL if the route has no "
        "trips going the reverse direction at all."
    ),
    "route_reverse_direction_n_trips": (
        "Row count backing the average above (NULL, not 0, when the average is NULL)."
    ),
    "route_reverse_direction_hour_avg_trip_duration_seconds": (
        "Same as route_reverse_direction_avg_trip_duration_seconds, "
        "additionally restricted to the opposite-direction trips whose "
        "trip_hour matches this row's own trip_hour."
    ),
    "route_reverse_direction_hour_n_trips": "Row count backing the average above.",
}

with conn.cursor() as cur:
    for col, text in METRICS_COMMENTS.items():
        comment_on_column(cur, "trip_validity_trip_metrics", col, text)

conn.execute(
    sql.SQL("COMMENT ON TABLE ml.trip_validity_trip_metrics IS {};").format(
        sql.Literal(
            "Trip Validity model: one feature row per trip, built entirely from "
            "ml.trip_validity_trips/trip_fares/route_gtfs_match/route_shapes/"
            "route_schedule - never touches silver. See "
            "ml/trip_validity_model/notebooks/03_trip_metrics.ipynb."
        )
    )
)
conn.commit()
print("comments applied")

comments applied


## Indexes and verification

In [14]:
import polars as pl

conn.execute("""
    CREATE INDEX trip_validity_trip_metrics_route_id_idx
        ON ml.trip_validity_trip_metrics (route_id);
    CREATE INDEX trip_validity_trip_metrics_route_direction_idx
        ON ml.trip_validity_trip_metrics (route_id, route_direction);
    ANALYZE ml.trip_validity_trip_metrics;
""")
conn.commit()

EXPECTED_TRIP_COUNT = 940988  # ml.trip_validity_trips row count

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            count(*) AS rows,
            count(*) FILTER (WHERE trip_distance_meters IS NOT NULL) AS has_distance,
            count(*) FILTER (
                WHERE trip_points_standard_distance_meters IS NOT NULL
            ) AS has_cohesion,
            count(*) FILTER (
                WHERE gtfs_feed_version_date IS NOT NULL
            ) AS has_gtfs_match,
            count(*) FILTER (
                WHERE path_frechet_distance_to_i_meters IS NOT NULL
                   OR path_frechet_distance_to_v_meters IS NOT NULL
            ) AS has_frechet,
            count(*) FILTER (
                WHERE route_i_scheduled_duration_avg_seconds IS NOT NULL
                   OR route_v_scheduled_duration_avg_seconds IS NOT NULL
            ) AS has_sched_avg,
            count(*) FILTER (
                WHERE route_avg_trip_duration_seconds_loo IS NOT NULL
            ) AS has_route_loo,
            count(*) FILTER (
                WHERE trip_progress_correlation_to_i IS NOT NULL
                   OR trip_progress_correlation_to_v IS NOT NULL
            ) AS has_progress_corr,
            count(*) FILTER (
                WHERE route_reverse_direction_avg_trip_duration_seconds IS NOT NULL
            ) AS has_reverse_dir
        FROM ml.trip_validity_trip_metrics;
    """)
    cols = [d.name for d in cur.description]
    row = cur.fetchone()

summary = pl.DataFrame([dict(zip(cols, row, strict=True))])
print(summary)
if summary["rows"][0] != EXPECTED_TRIP_COUNT:
    msg = "row count mismatch against trip_validity_trips"
    raise AssertionError(msg)
print("OK: row count matches trip_validity_trips")

shape: (1, 9)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ rows   ┆ has_distan ┆ has_cohesi ┆ has_gtfs_ ┆ … ┆ has_sched ┆ has_route ┆ has_progr ┆ has_rever │
│ ---    ┆ ce         ┆ on         ┆ match     ┆   ┆ _avg      ┆ _loo      ┆ ess_corr  ┆ se_dir    │
│ i64    ┆ ---        ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│        ┆ i64        ┆ i64        ┆ i64       ┆   ┆ i64       ┆ i64       ┆ i64       ┆ i64       │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 940988 ┆ 899299     ┆ 899299     ┆ 934155    ┆ … ┆ 934155    ┆ 940980    ┆ 605112    ┆ 897435    │
└────────┴────────────┴────────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘
OK: row count matches trip_validity_trips
